# Faster R-CNN with ResNet-50-FPN backbone — fair RDD2022 comparison

COCO-pretrained `fasterrcnn_resnet50_fpn_v2` (torchvision), new head for the 4 damage classes + background (5 outputs, torchvision convention), longest image side 640 px (aspect ratio kept), base LR 0.02, batch 8 × 2 accumulation = 16.

**Everything is saved to Drive** under `MyDrive/RDD_fair/results/fasterrcnn_r50_fpn_v2/seed0/`
(`checkpoints/last.pt` every epoch, `checkpoints/best.pt` = selected epoch, `logs/train_log.csv`, `logs/run_info.json`).

**If Colab disconnects:** reconnect and *Run all* again. A finished run is skipped; an unfinished run resumes from its last saved epoch.

In [ ]:
# 1) Mount Drive and import the shared code (upload the folder rdd_fair/ to MyDrive/RDD_fair/code/)
from google.colab import drive
drive.mount('/content/drive')
import sys, os
CODE_DIR = '/content/drive/MyDrive/RDD_fair/code'
sys.path.insert(0, CODE_DIR)
!nvidia-smi

In [ ]:
# 2) Same library versions in every session, same GPU type, same dataset
from rdd_fair import config as C, utils as U, prepare_data as P
U.pin_or_install_packages()
C.ensure_dirs()
U.check_gpu_consistency()
data_yaml = P.prepare_rdd()

In [ ]:
# 3) Train
from rdd_fair import tv_train as T
MODEL = 'fasterrcnn_r50_fpn_v2'
print(C.MODEL_CFG[MODEL])
for seed in C.SEEDS:
    T.train(MODEL, seed)

In [ ]:
# 4) Training curves (read back from Drive)
import pandas as pd, matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for seed in C.SEEDS:
    p = os.path.join(C.run_dir(MODEL, seed, 'logs'), 'train_log.csv')
    if os.path.exists(p):
        d = pd.read_csv(p).drop_duplicates('epoch', keep='last')
        ax[0].plot(d.epoch, d.train_loss, label=f'seed {seed}')
        ax[1].plot(d.epoch, d.val_mAP50_95, label=f'seed {seed} mAP50-95')
        ax[1].plot(d.epoch, d.val_mAP50, '--', label=f'seed {seed} mAP50')
ax[0].set_title('train loss'); ax[1].set_title('val mAP (fair_eval)'); ax[1].legend(fontsize=8)
plt.show()
for seed in C.SEEDS:
    print(seed, U.load_json(os.path.join(C.run_dir(MODEL, seed, 'logs'), 'selection.json')))